# Feature Fusion Klinis + PVC ViT

Notebook ini melanjutkan pipeline penelitian setelah inference YOLO PVC. Tujuannya membuat vektor 1D per pasien dari data klinis, top-5 PVC confidence model Medium, dan embedding gambar cycle PVC menggunakan Vision Transformer. Backbone gambar dibuat modular sehingga nanti bisa diganti ke CNN-based extractor.


In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torchvision import transforms as T
from torchvision.models import ResNet18_Weights, ViT_B_16_Weights, resnet18, vit_b_16

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebook-preprocess" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CLINICAL_DATA_PATH = DATA_DIR / "PENELITIAN PVC TELKOM - Sheet1.csv"
VALID_PATIENTS_FILE = PROJECT_ROOT / "valid-patient.txt"
FEATURES_ROOT = DATA_DIR / "ecg-data-features"
FUSION_OUTPUT_DIR = FEATURES_ROOT / "fusion-ready"
FUSION_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_FOR_PVC_SELECTION = "yolo12_medium"
TOP_K_PVC = 5

# Default False agar notebook bisa berjalan offline. Ubah True jika ingin memakai ImageNet pretrained weights.
USE_PRETRAINED_IMAGE_BACKBONE = False
IMAGE_BACKBONE = "vit_b_16"  # opsi: "vit_b_16", "resnet18"
IMAGE_SIZE = 224
L2_NORMALIZE_IMAGE_EMBEDDING = True
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

with open(VALID_PATIENTS_FILE, "r") as f:
    PATIENT_IDS = [line.strip() for line in f if line.strip()]

print(f"Project root : {PROJECT_ROOT}")
print(f"Patients     : {PATIENT_IDS}")
print(f"Device       : {DEVICE}")
print(f"Backbone     : {IMAGE_BACKBONE}, pretrained={USE_PRETRAINED_IMAGE_BACKBONE}")


Project root : /home/nugee/code-program/code-thesis/thesis-severity
Patients     : ['P-00001', 'P-00002', 'P-00007', 'P-00015']
Device       : cuda
Backbone     : vit_b_16, pretrained=False


## 1. Load dan rapikan data klinis

CSV memiliki header bertingkat: baris pertama berisi sub-kolom seperti Hipertensi, Diabetes, CAD, LVEF, LVIDd, TAPSE, Percent, dan Run VT. Cell ini menamai ulang kolom yang dibutuhkan, mengubah nilai kosong menjadi 0, lalu membuat fitur numerik 1D.


In [2]:
CLINICAL_COLUMN_MAP = {
    "ID-Pasien": "patient_id",
    "Faktor Risiko": "risk_hipertensi",
    "Unnamed: 5": "risk_diabetes",
    "Unnamed: 6": "risk_cad",
    "Pingsan": "syncope",
    "Katerisasi \nCoroner": "coronary_catheterization",
    "Echo": "lvef",
    "Unnamed: 10": "lvidd",
    "Unnamed: 11": "tapse",
    "Unnamed: 12": "rv_dilatation",
    "Unnamed: 13": "diastolic_function",
    "Holter (PVC%)": "holter_pvc_percent",
    "Unnamed: 15": "run_vt",
}

CLINICAL_FEATURE_COLUMNS = [
    "risk_hipertensi",
    "risk_diabetes",
    "risk_cad",
    "syncope",
    "coronary_catheterization",
    "lvef",
    "lvidd",
    "tapse",
    "rv_dilatation",
    "diastolic_function_grade",
    "holter_pvc_percent",
    "run_vt",
]


def clean_text(value: Any) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip().replace("\n", " ").strip()


def parse_number(value: Any) -> float:
    text = clean_text(value).replace(",", ".")
    if text == "" or text.lower() in {"nan", "none", "-", "tidak ada"}:
        return 0.0
    match = re.search(r"-?\d+(?:\.\d+)?", text.replace("%", ""))
    return float(match.group(0)) if match else 0.0


def parse_binary(value: Any) -> float:
    text = clean_text(value).lower()
    if text in {"true", "ya", "yes", "y", "1", "ada", "positif"}:
        return 1.0
    if text in {"false", "tidak", "no", "n", "0", "", "-", "nan", "none"}:
        return 0.0
    return 1.0 if any(token in text for token in ["ya", "true", "ada", "positif"]) else 0.0


def parse_coronary_catheterization(value: Any) -> float:
    text = clean_text(value).lower()
    if text == "" or text == "-" or "blm" in text or "belum" in text or "tidak" in text:
        return 0.0
    return 1.0


def parse_diastolic_grade(value: Any) -> float:
    text = clean_text(value).upper()
    match = re.search(r"G\s*(\d+)", text)
    return float(match.group(1)) if match else 0.0


def load_clinical_features() -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    raw = pd.read_csv(CLINICAL_DATA_PATH)
    data = raw.iloc[1:].copy().rename(columns=CLINICAL_COLUMN_MAP)
    data = data[data["patient_id"].notna()].copy()
    data["patient_id"] = data["patient_id"].astype(str).str.strip()
    data = data[data["patient_id"].isin(PATIENT_IDS)].copy()

    features = pd.DataFrame({"patient_id": data["patient_id"]})
    features["risk_hipertensi"] = data["risk_hipertensi"].apply(parse_binary)
    features["risk_diabetes"] = data["risk_diabetes"].apply(parse_binary)
    features["risk_cad"] = data["risk_cad"].apply(parse_binary)
    features["syncope"] = data["syncope"].apply(parse_binary)
    features["coronary_catheterization"] = data["coronary_catheterization"].apply(parse_coronary_catheterization)
    features["lvef"] = data["lvef"].apply(parse_number)
    features["lvidd"] = data["lvidd"].apply(parse_number)
    features["tapse"] = data["tapse"].apply(parse_number)
    features["rv_dilatation"] = data["rv_dilatation"].apply(parse_binary)
    features["diastolic_function_grade"] = data["diastolic_function"].apply(parse_diastolic_grade)
    features["holter_pvc_percent"] = data["holter_pvc_percent"].apply(parse_number)
    features["run_vt"] = data["run_vt"].apply(parse_binary)

    # Pastikan semua pasien valid tetap ada; pasien tanpa data klinis akan menjadi 0.
    features = pd.DataFrame({"patient_id": PATIENT_IDS}).merge(features, on="patient_id", how="left")
    features[CLINICAL_FEATURE_COLUMNS] = features[CLINICAL_FEATURE_COLUMNS].fillna(0.0).astype(float)

    normalized = features.copy()
    normalization_params = {}
    for column in CLINICAL_FEATURE_COLUMNS:
        min_value = float(features[column].min())
        max_value = float(features[column].max())
        if max_value > min_value:
            normalized[column] = (features[column] - min_value) / (max_value - min_value)
        else:
            normalized[column] = 0.0
        normalization_params[column] = {"min": min_value, "max": max_value}

    return features, normalized, normalization_params


clinical_raw, clinical_normalized, normalization_params = load_clinical_features()
clinical_raw.to_csv(FUSION_OUTPUT_DIR / "clinical_features_raw.csv", index=False)
clinical_normalized.to_csv(FUSION_OUTPUT_DIR / "clinical_features_normalized.csv", index=False)
(FUSION_OUTPUT_DIR / "clinical_normalization_params.json").write_text(json.dumps(normalization_params, indent=2))

clinical_normalized


,patient_id,risk_hipertensi,risk_diabetes,risk_cad,syncope,coronary_catheterization,lvef,lvidd,tapse,rv_dilatation,diastolic_function_grade,holter_pvc_percent,run_vt
0,P-00001,0.0,0.0,0.0,0.0,0.0,0.4375,0.723982,0.529412,1.0,0.0,0.000000,0.0
1,P-00002,0.0,0.0,0.0,0.0,0.0,0.1875,1.000000,0.688235,0.0,1.0,0.222222,0.0
2,P-00007,1.0,0.0,0.0,0.0,0.0,0.0000,0.000000,1.000000,1.0,0.0,1.000000,0.0
3,P-00015,1.0,0.0,0.0,0.0,0.0,1.0000,0.000000,0.000000,0.0,0.0,0.611111,0.0


## 2. Ambil top-5 PVC dari model YOLO Medium

Untuk setiap pasien, dipilih maksimal 5 cycle PVC dengan confidence tertinggi dari folder `data/ecg-data-features/{patient}/yolo12_medium/detections.csv`. Jika PVC kurang dari 5, hanya data yang tersedia yang dipakai; slot sisanya akan dipadding 0 pada tahap fusion.


In [3]:
def load_top_pvc_detections(patient_id: str, top_k: int = TOP_K_PVC) -> pd.DataFrame:
    detection_path = FEATURES_ROOT / patient_id / MODEL_FOR_PVC_SELECTION / "detections.csv"
    if not detection_path.exists():
        return pd.DataFrame()

    detections = pd.read_csv(detection_path)
    if detections.empty or "class_name" not in detections.columns:
        return pd.DataFrame()

    pvc = detections[detections["class_name"].astype(str).str.lower() == "pvc"].copy()
    if pvc.empty:
        return pvc

    pvc["confidence"] = pd.to_numeric(pvc["confidence"], errors="coerce").fillna(0.0)
    pvc = pvc.sort_values("confidence", ascending=False)
    pvc = pvc.drop_duplicates(subset=["cycle_file_name"], keep="first")
    pvc = pvc.head(top_k).copy()
    pvc.insert(0, "pvc_rank", range(1, len(pvc) + 1))
    return pvc


selected_rows = []
for patient_id in PATIENT_IDS:
    top_pvc = load_top_pvc_detections(patient_id)
    if top_pvc.empty:
        selected_rows.append({
            "patient_id": patient_id,
            "pvc_rank": 0,
            "cycle_id": "",
            "cycle_file_name": "",
            "cycle_file_path": "",
            "confidence": 0.0,
            "note": "no_medium_pvc_detection",
        })
        continue
    for _, row in top_pvc.iterrows():
        selected_rows.append({
            "patient_id": patient_id,
            "pvc_rank": int(row["pvc_rank"]),
            "cycle_id": row.get("cycle_id", ""),
            "cycle_file_name": row.get("cycle_file_name", ""),
            "cycle_file_path": row.get("cycle_file_path", ""),
            "lead_label": row.get("lead_label", ""),
            "cycle_order": row.get("cycle_order", ""),
            "confidence": float(row.get("confidence", 0.0)),
            "bbox_yolo_x1": row.get("bbox_yolo_x1", ""),
            "bbox_yolo_y1": row.get("bbox_yolo_y1", ""),
            "bbox_yolo_x2": row.get("bbox_yolo_x2", ""),
            "bbox_yolo_y2": row.get("bbox_yolo_y2", ""),
            "bbox_raw_x1": row.get("bbox_raw_x1", ""),
            "bbox_raw_y1": row.get("bbox_raw_y1", ""),
            "bbox_raw_x2": row.get("bbox_raw_x2", ""),
            "bbox_raw_y2": row.get("bbox_raw_y2", ""),
            "note": "selected_medium_pvc",
        })

selected_medium_pvc = pd.DataFrame(selected_rows)
selected_medium_pvc.to_csv(FUSION_OUTPUT_DIR / "selected_medium_pvc_top5.csv", index=False)
selected_medium_pvc


,patient_id,pvc_rank,cycle_id,cycle_file_name,cycle_file_path,lead_label,cycle_order,confidence,bbox_yolo_x1,bbox_yolo_y1,bbox_yolo_x2,bbox_yolo_y2,bbox_raw_x1,bbox_raw_y1,bbox_raw_x2,bbox_raw_y2,note
0,P-00001,1,P-00001_11_V5_cycle_02,11_V5_cycle_02.png,/home/nugee/code-program/code-thesis/thesis-se...,V5,2.0,0.922575,189.050110,132.343567,594.359497,571.061768,960.573124,471.857779,1033.866572,563.044082,selected_medium_pvc
1,P-00001,2,P-00001_10_V4_cycle_01,10_V4_cycle_01.png,/home/nugee/code-program/code-thesis/thesis-se...,V4,1.0,0.810193,197.961853,90.006653,640.000000,528.509888,739.036331,378.592477,818.971563,458.098999,selected_medium_pvc
2,P-00001,3,P-00001_12_V6_cycle_02,12_V6_cycle_02.png,/home/nugee/code-program/code-thesis/thesis-se...,V6,2.0,0.744392,200.946777,111.661133,635.492310,582.043457,962.724438,565.630443,1041.304755,659.237719,selected_medium_pvc
3,P-00001,4,P-00001_04_aVR_cycle_02,04_aVR_cycle_02.png,/home/nugee/code-program/code-thesis/thesis-se...,aVR,2.0,0.614169,169.390320,0.000000,640.000000,556.093933,435.463302,390.575646,516.842010,491.402940,selected_medium_pvc
4,P-00002,1,P-00002_07_V1_cycle_01,07_V1_cycle_01.png,/home/nugee/code-program/code-thesis/thesis-se...,V1,1.0,0.843592,0.000000,0.000000,638.507629,493.506042,493.631312,92.988930,582.064619,130.334795,selected_medium_pvc
5,P-00002,2,P-00002_05_aVL_cycle_02,05_aVL_cycle_02.png,/home/nugee/code-program/code-thesis/thesis-se...,aVL,2.0,0.752293,251.191788,46.888000,415.864868,436.002197,255.268948,319.188498,277.220899,395.748241,selected_medium_pvc
6,P-00002,3,P-00002_12_V6_cycle_01,12_V6_cycle_01.png,/home/nugee/code-program/code-thesis/thesis-se...,V6,1.0,0.640501,245.218170,115.714600,388.955719,631.523926,527.594029,384.708019,547.501680,467.459371,selected_medium_pvc
7,P-00007,1,P-00007_03_III_cycle_05,03_III_cycle_05.png,/home/nugee/code-program/code-thesis/thesis-se...,III,5.0,0.864976,101.440674,160.227295,640.000000,582.620178,322.116442,244.230545,389.934525,319.508243,selected_medium_pvc
8,P-00007,2,P-00007_08_V2_cycle_03,08_V2_cycle_03.png,/home/nugee/code-program/code-thesis/thesis-se...,V2,3.0,0.862176,175.539291,1.091797,634.950562,422.657166,604.727654,159.827705,665.352713,215.833870,selected_medium_pvc
9,P-00007,3,P-00007_04_aVR_cycle_05,04_aVR_cycle_05.png,/home/nugee/code-program/code-thesis/thesis-se...,aVR,5.0,0.858120,139.081039,0.000000,640.000000,361.294983,326.856305,308.996310,389.934525,338.264037,selected_medium_pvc


## 3. Backbone ekstraksi fitur gambar PVC

Default menggunakan `torchvision.models.vit_b_16` dengan head diganti `Identity`, sehingga outputnya adalah embedding 768 dimensi. Jika ingin eksperimen CNN, ubah `IMAGE_BACKBONE = "resnet18"` pada cell konfigurasi.


In [4]:
def build_image_feature_extractor(backbone_name: str):
    backbone_name = backbone_name.lower()
    if backbone_name == "vit_b_16":
        weights = ViT_B_16_Weights.DEFAULT if USE_PRETRAINED_IMAGE_BACKBONE else None
        model = vit_b_16(weights=weights)
        model.heads = torch.nn.Identity()
        if weights is not None:
            preprocess = weights.transforms()
        else:
            preprocess = T.Compose([
                T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
        embedding_dim = 768
    elif backbone_name == "resnet18":
        weights = ResNet18_Weights.DEFAULT if USE_PRETRAINED_IMAGE_BACKBONE else None
        model = resnet18(weights=weights)
        model.fc = torch.nn.Identity()
        if weights is not None:
            preprocess = weights.transforms()
        else:
            preprocess = T.Compose([
                T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                T.ToTensor(),
                T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ])
        embedding_dim = 512
    else:
        raise ValueError(f"Unsupported IMAGE_BACKBONE: {backbone_name}")

    model = model.to(DEVICE).eval()
    return model, preprocess, embedding_dim


image_model, image_preprocess, IMAGE_EMBEDDING_DIM = build_image_feature_extractor(IMAGE_BACKBONE)
print(f"Embedding dim: {IMAGE_EMBEDDING_DIM}")


def extract_image_embedding(image_path: str | Path) -> np.ndarray:
    image_path = Path(image_path)
    if not image_path.exists():
        return np.zeros(IMAGE_EMBEDDING_DIM, dtype=np.float32)

    image = Image.open(image_path).convert("RGB")
    tensor = image_preprocess(image).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        embedding = image_model(tensor).squeeze(0).detach().cpu().numpy().astype(np.float32)

    if L2_NORMALIZE_IMAGE_EMBEDDING:
        norm = float(np.linalg.norm(embedding))
        if norm > 0:
            embedding = embedding / norm
    return embedding


Embedding dim: 768


## 4. Bentuk vektor fusion per pasien

Setiap pasien mendapat satu vektor 1D:

- clinical normalized features
- `pvc_conf_top_1` sampai `pvc_conf_top_5`
- embedding gambar PVC per slot top-5, dipadding 0 jika PVC kurang dari 5

Output disimpan sebagai CSV gabungan dan `.npy` per pasien.


In [5]:
def patient_selected_pvc(patient_id: str) -> pd.DataFrame:
    rows = selected_medium_pvc[
        (selected_medium_pvc["patient_id"] == patient_id)
        & (selected_medium_pvc["pvc_rank"].astype(int) > 0)
    ].copy()
    if rows.empty:
        return rows
    return rows.sort_values("pvc_rank").head(TOP_K_PVC)


fusion_rows = []
embedding_rows = []
feature_name_rows = []

clinical_lookup = clinical_normalized.set_index("patient_id")

clinical_feature_names = CLINICAL_FEATURE_COLUMNS
confidence_feature_names = [f"pvc_conf_top_{rank}" for rank in range(1, TOP_K_PVC + 1)]
embedding_feature_names = [
    f"pvc_{rank}_embed_{dim:04d}"
    for rank in range(1, TOP_K_PVC + 1)
    for dim in range(IMAGE_EMBEDDING_DIM)
]
all_feature_names = clinical_feature_names + confidence_feature_names + embedding_feature_names

for idx, name in enumerate(all_feature_names):
    if name in clinical_feature_names:
        source = "clinical"
    elif name in confidence_feature_names:
        source = "medium_yolo_confidence"
    else:
        source = IMAGE_BACKBONE
    feature_name_rows.append({"feature_index": idx, "feature_name": name, "source": source})

for patient_id in PATIENT_IDS:
    clinical_vector = clinical_lookup.loc[patient_id, clinical_feature_names].to_numpy(dtype=np.float32)
    pvc_rows = patient_selected_pvc(patient_id)

    confidence_slots = np.zeros(TOP_K_PVC, dtype=np.float32)
    embedding_slots = np.zeros((TOP_K_PVC, IMAGE_EMBEDDING_DIM), dtype=np.float32)

    for _, pvc_row in pvc_rows.iterrows():
        slot = int(pvc_row["pvc_rank"]) - 1
        if slot < 0 or slot >= TOP_K_PVC:
            continue
        confidence_slots[slot] = float(pvc_row.get("confidence", 0.0))
        embedding = extract_image_embedding(pvc_row.get("cycle_file_path", ""))
        embedding_slots[slot] = embedding
        embedding_rows.append({
            "patient_id": patient_id,
            "pvc_rank": slot + 1,
            "cycle_id": pvc_row.get("cycle_id", ""),
            "cycle_file_name": pvc_row.get("cycle_file_name", ""),
            "cycle_file_path": pvc_row.get("cycle_file_path", ""),
            "confidence": float(pvc_row.get("confidence", 0.0)),
            "embedding_dim": IMAGE_EMBEDDING_DIM,
            "embedding_norm": float(np.linalg.norm(embedding)),
        })

    fusion_vector = np.concatenate([clinical_vector, confidence_slots, embedding_slots.reshape(-1)]).astype(np.float32)

    patient_fusion_dir = FEATURES_ROOT / patient_id / "fusion"
    patient_fusion_dir.mkdir(parents=True, exist_ok=True)
    np.save(patient_fusion_dir / "clinical_vector_normalized.npy", clinical_vector)
    np.save(patient_fusion_dir / "pvc_confidence_top5.npy", confidence_slots)
    np.save(patient_fusion_dir / f"pvc_{IMAGE_BACKBONE}_top5_embeddings.npy", embedding_slots)
    np.save(patient_fusion_dir / "fusion_vector.npy", fusion_vector)
    (patient_fusion_dir / "fusion_feature_names.json").write_text(json.dumps(all_feature_names, indent=2))

    row = {"patient_id": patient_id, "selected_pvc_count": int(len(pvc_rows)), "fusion_vector_dim": int(len(fusion_vector))}
    row.update({name: float(value) for name, value in zip(all_feature_names, fusion_vector)})
    fusion_rows.append(row)

fusion_features = pd.DataFrame(fusion_rows)
feature_names = pd.DataFrame(feature_name_rows)
embedding_manifest = pd.DataFrame(embedding_rows)

fusion_features.to_csv(FUSION_OUTPUT_DIR / "fusion_features.csv", index=False)
feature_names.to_csv(FUSION_OUTPUT_DIR / "fusion_feature_names.csv", index=False)
embedding_manifest.to_csv(FUSION_OUTPUT_DIR / "pvc_image_embedding_manifest.csv", index=False)
np.save(FUSION_OUTPUT_DIR / "fusion_matrix.npy", fusion_features[all_feature_names].to_numpy(dtype=np.float32))

metadata = {
    "patient_ids": PATIENT_IDS,
    "clinical_data_path": str(CLINICAL_DATA_PATH),
    "selected_yolo_model": MODEL_FOR_PVC_SELECTION,
    "top_k_pvc": TOP_K_PVC,
    "image_backbone": IMAGE_BACKBONE,
    "use_pretrained_image_backbone": USE_PRETRAINED_IMAGE_BACKBONE,
    "image_embedding_dim": IMAGE_EMBEDDING_DIM,
    "l2_normalize_image_embedding": L2_NORMALIZE_IMAGE_EMBEDDING,
    "clinical_feature_count": len(clinical_feature_names),
    "confidence_feature_count": len(confidence_feature_names),
    "embedding_feature_count": len(embedding_feature_names),
    "fusion_vector_dim": len(all_feature_names),
    "outputs": {
        "fusion_features_csv": str(FUSION_OUTPUT_DIR / "fusion_features.csv"),
        "fusion_matrix_npy": str(FUSION_OUTPUT_DIR / "fusion_matrix.npy"),
        "feature_names_csv": str(FUSION_OUTPUT_DIR / "fusion_feature_names.csv"),
        "selected_medium_pvc_csv": str(FUSION_OUTPUT_DIR / "selected_medium_pvc_top5.csv"),
    },
}
(FUSION_OUTPUT_DIR / "fusion_metadata.json").write_text(json.dumps(metadata, indent=2))

print(f"Fusion vector dim : {len(all_feature_names)}")
print(f"Saved to          : {FUSION_OUTPUT_DIR}")
fusion_features[["patient_id", "selected_pvc_count", "fusion_vector_dim"] + confidence_feature_names]


Fusion vector dim : 3857
Saved to          : /home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready


,patient_id,selected_pvc_count,fusion_vector_dim,pvc_conf_top_1,pvc_conf_top_2,pvc_conf_top_3,pvc_conf_top_4,pvc_conf_top_5
0,P-00001,4,3857,0.922575,0.810193,0.744392,0.614169,0.000000
1,P-00002,3,3857,0.843592,0.752293,0.640501,0.000000,0.000000
2,P-00007,5,3857,0.864976,0.862176,0.858120,0.757393,0.730365
3,P-00015,0,3857,0.000000,0.000000,0.000000,0.000000,0.000000


## 5. Output yang digunakan tahap berikutnya

Gunakan `data/ecg-data-features/fusion-ready/fusion_features.csv` atau `fusion_matrix.npy` sebagai input model fusion. Jika ingin membaca per pasien, gunakan folder `data/ecg-data-features/{patient}/fusion/`.


In [6]:
print("Global outputs:")
for path in sorted(FUSION_OUTPUT_DIR.glob("*")):
    print(path)

print("\nPer-patient fusion outputs:")
for patient_id in PATIENT_IDS:
    fusion_dir = FEATURES_ROOT / patient_id / "fusion"
    print(patient_id, "->", fusion_dir)


Global outputs:
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/clinical_features_normalized.csv
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/clinical_features_raw.csv
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/clinical_normalization_params.json
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/fusion_feature_names.csv
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/fusion_features.csv
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/fusion_matrix.npy
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/fusion_metadata.json
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg-data-features/fusion-ready/pvc_image_embedding_manifest.csv
/home/nugee/code-program/code-thesis/thesis-severity/data/ecg